### Initialize Intent Resolution Evaluator


In [2]:
import os
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.ai.evaluation import IntentResolutionEvaluator
from pprint import pprint
from dotenv import load_dotenv

load_dotenv()

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint="https://semantic-aifoundry.cognitiveservices.azure.com/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2025-01-01-preview",
    azure_deployment="gpt-4o",
)

intent_resolution_evaluator = IntentResolutionEvaluator(model_config)

Class IntentResolutionEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
# Success example. Intent is identified and understood and the response correctly resolves user intent
result = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response="Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.",
)
pprint(result)

Conversation history could not be parsed, falling back to original query: What are the opening hours of the Eiffel Tower?
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.


{'intent_resolution': 5.0,
 'intent_resolution_reason': 'User wanted to know the opening hours of the '
                             'Eiffel Tower. Agent provided a clear and '
                             'accurate response with the correct hours, fully '
                             "resolving the user's intent without any gaps or "
                             'issues.',
 'intent_resolution_result': 'pass',
 'intent_resolution_threshold': 3}


In [4]:
# Failure example. Even though intent is correctly identified, the response does not resolve the user intent
result = intent_resolution_evaluator(
    query="What is the opening hours of the Eiffel Tower?",
    response="Please check the official website for the up-to-date information on Eiffel Tower opening hours.",
)
pprint(result)

Conversation history could not be parsed, falling back to original query: What is the opening hours of the Eiffel Tower?
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Please check the official website for the up-to-date information on Eiffel Tower opening hours.


{'intent_resolution': 2.0,
 'intent_resolution_reason': 'User wanted the opening hours of the Eiffel '
                             'Tower. Agent redirected them to the official '
                             'website without providing the requested '
                             'information directly, leaving the intent '
                             'partially unresolved.',
 'intent_resolution_result': 'fail',
 'intent_resolution_threshold': 3}


### Conversation History 

In [ ]:
query = [
    {"role": "system", "content": "You are a friendly and helpful customer service agent."},
    {
        "createdAt": "2025-03-14T06:14:20Z",
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Hi, I need help with the last 2 orders on my account #888. Could you please update me on their status?",
            }
        ],
    },
]

response = [
    {
        "createdAt": "2025-03-14T06:14:30Z",
        "run_id": "0",
        "role": "assistant",
        "content": [{"type": "text", "text": "Hello! Let me quickly look up your account details."}],
    },
    {
        "createdAt": "2025-03-14T06:14:35Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_001",
                "name": "get_orders",
                "arguments": {"account_number": "888"},
            }
        ],
    },
    {
        "createdAt": "2025-03-14T06:14:40Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_001",
        "role": "tool",
        "content": [{"type": "tool_result", "tool_result": '[{ "order_id": "123" }, { "order_id": "124" }]'}],
    },
    {
        "createdAt": "2025-03-14T06:14:45Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Thanks for your patience. I see two orders on your account. Let me fetch the details for both.",
            }
        ],
    },
    {
        "createdAt": "2025-03-14T06:14:50Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_002",
                "name": "get_order",
                "arguments": {"order_id": "123"},
            },
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_003",
                "name": "get_order",
                "arguments": {"order_id": "124"},
            },
        ],
    },
    {
        "createdAt": "2025-03-14T06:14:55Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_002",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": '{ "order": { "id": "123", "status": "shipped", "delivery_date": "2025-03-15" } }',
            }
        ],
    },
    {
        "createdAt": "2025-03-14T06:15:00Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_003",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": '{ "order": { "id": "124", "status": "delayed", "expected_delivery": "2025-03-20" } }',
            }
        ],
    },
    {
        "createdAt": "2025-03-14T06:15:05Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "The order with ID 123 has been shipped and is expected to be delivered on March 15, 2025. However, the order with ID 124 is delayed and should now arrive by March 20, 2025. Is there anything else I can help you with?",
            }
        ],
    },
]

# please note that the tool definitions are not strictly required, and that some of the tools below are not used in the example above and that is ok.
# if context length is a concern you can remove the unused tool definitions or even the tool definitions altogether as the impact to the intent resolution evaluation is usual minimal.
tool_definitions = [
    {
        "name": "get_orders",
        "description": "Get the list of orders for a given account number.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {"type": "string", "description": "The account number to get the orders for."}
            },
        },
    },
    {
        "name": "get_order",
        "description": "Get the details of a specific order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "The order ID to get the details for."}},
        },
    },
    {
        "name": "initiate_return",
        "description": "Initiate the return process for an order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "The order ID for the return process."}},
        },
    },
    {
        "name": "update_shipping_address",
        "description": "Update the shipping address for a given account.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {"type": "string", "description": "The account number to update."},
                "new_address": {"type": "string", "description": "The new shipping address."},
            },
        },
    },
]

result = intent_resolution_evaluator(
    query=query,
    response=response,
    tool_definitions=tool_definitions,
)
pprint(result)

{'intent_resolution': 2.0,
 'intent_resolution_reason': 'User wanted an update on the status of their '
                             'last two orders associated with account #888. '
                             'The agent provided conflicting information, '
                             'stating no orders exist on the account but then '
                             'detailing the status of two orders. This '
                             'inconsistency undermines the resolution.',
 'intent_resolution_result': 'fail',
 'intent_resolution_threshold': 3}


### Visualize in the AI Foundry 

In [ ]:
from azure.ai.evaluation import evaluate
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# This sample files contains the evaluation data in JSONL format. Where each line is a run from agent.
# This was saved using agent thread and converter.
file_name = "./data/evaluation_data.jsonl"

project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_AGENT_ENDPOINT"],
    credential=DefaultAzureCredential()
)


response = evaluate(
    data=file_name,
    evaluation_name="Intent Resolution Evaluation",
    evaluators={
        "intent_resolution": intent_resolution_evaluator,
    },
   azure_ai_project=os.environ.get("AZURE_AI_AGENT_ENDPOINT"),
)
pprint(f'AI Foundary URL: {response.get("studio_url")}')

EvaluationException: (UserError) The 'azure_ai_project' parameter must be a dictionary.